In [7]:
# logistic_summary.py
# Chạy trong Python environment (>=3.7). Cần cài: pandas, numpy, statsmodels, scikit-learn
# pip install pandas numpy statsmodels scikit-learn

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.impute import SimpleImputer

# ---------- Config ----------
PATH = "/Users/macbook/Downloads/Binary.xlsx"   # sửa đường dẫn tới file Excel/CSV
SHEET = 0                        # sheet index or name
target_col = "TraNo"             # tên cột target (0/1 hoặc 1..5 (sẽ map))
# Các biến độc lập (ví dụ): nếu data bạn có đúng các cột này thì dùng; nếu khác thay tên cho phù hợp.
feature_cols = ["HocVan", "Tuoi", "ThuNhap", "HoKhau"]
# Nếu bạn muốn cho tự động dùng numeric + vài categorical nhỏ, comment out feature_cols và dùng auto selection code below.
# -----------------------------

# 1. Load data
if PATH.lower().endswith('.xlsx') or PATH.lower().endswith('.xls'):
    df = pd.read_excel(PATH, sheet_name=SHEET)
else:
    df = pd.read_csv(PATH)

# 2. Basic target processing
if target_col not in df.columns:
    raise ValueError(f"Không tìm thấy cột target '{target_col}' trong file. Kiểm tra lại tên cột.")

y = df[target_col].copy()

# Nếu target là ordinal (1..5), map về binary (ví dụ >=4 -> default=1)
if pd.api.types.is_integer_dtype(y) and y.nunique() > 2:
    # Ví dụ mapping: nhóm 1-3 = good (0), 4-5 = bad (1). Chỉnh theo nghiệp vụ của bạn.
    y = y.apply(lambda v: 1 if v >= 4 else 0)

# 3. Prepare X (features)
# if explicit features not present, auto select
missing_feats = [c for c in feature_cols if c not in df.columns]
if missing_feats:
    # Fallback: choose numeric + up to 3 categorical low-cardinality
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    num = [c for c in num if c != target_col]
    cat = [c for c in df.select_dtypes(include=['object','category']).columns if df[c].nunique() <= 8]
    feature_cols = (num[:3] + cat[:3])[:4]  # up to 4 features
    print("Auto-selected features:", feature_cols)

X_raw = df[feature_cols].copy()

# 4. Impute numeric and encode categoricals
num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_raw.columns if c not in num_cols]

# Impute numeric
if num_cols:
    imputer = SimpleImputer(strategy='median')
    X_raw[num_cols] = imputer.fit_transform(X_raw[num_cols])

# Encode categoricals (one-hot, drop first to avoid multicollinearity)
if cat_cols:
    X_cat = pd.get_dummies(X_raw[cat_cols].astype(str), drop_first=True)
else:
    X_cat = pd.DataFrame(index=X_raw.index)

# Final design matrix
X = pd.concat([X_raw[num_cols], X_cat], axis=1)

# Drop rows with missing target or X (if any)
mask = (~y.isna()) & (X.notna().all(axis=1))
y = y.loc[mask]
X = X.loc[mask]

# Add constant for intercept
X_sm = sm.add_constant(X, has_constant='add')

# 5. Fit logistic regression (statsmodels Logit - MLE)
model = sm.Logit(y.astype(int), X_sm)
result = model.fit(disp=False)  # disp=False to suppress iteration output

# 6. Build output table: odds ratio + std err + z + p | 95% CI (odds)
params = result.params
bse = result.bse
z = result.tvalues
pvals = result.pvalues
conf = result.conf_int()  # in log-odds scale
conf.columns = ['2.5%', '97.5%']

# exponentiate to get odds ratios and CI on odds scale
odds_ratio = np.exp(params)
ci_lower = np.exp(conf['2.5%'])
ci_upper = np.exp(conf['97.5%'])

# assemble dataframe
summary_df = pd.DataFrame({
    "Odds ratio": odds_ratio.round(6),
    "Std. err.": bse.round(6),
    "z": z.round(3),
    "P>|z|": pvals.round(3),
    "[95% CI lower]": ci_lower.round(6),
    "[95% CI upper]": ci_upper.round(6)
})

# 7. Print regression-like summary similar to your example
print("\nLogistic regression results (odds ratios)\n")
print(f"Number of obs = {int(result.nobs)}")
llf = result.llf
llnull = result.llnull
lr_stat = result.llr   # likelihood ratio chi2
llr_pvalue = result.llr_pvalue
pseudo_r2 = result.prsquared
print(f"Log likelihood = {llf:.6f}")
print(f"LR chi2({result.df_model:.0f}) = {lr_stat:.2f}")
print(f"Prob > chi2 = {llr_pvalue:.6f}")
print(f"Pseudo R2 = {pseudo_r2:.4f}\n")

# Show table
print(summary_df.to_string())

# 8. Nicely formatted table (optional)
try:
    from tabulate import tabulate
    print("\n\nFormatted Table:\n")
    print(tabulate(summary_df.reset_index().rename(columns={'index':'Variable'}), headers='keys', tablefmt='github', showindex=False))
except Exception:
    pass

# 9. Interpretation hint
print("\nNote: Odds ratio >1 implies increased odds (higher risk) per unit increase in predictor; <1 implies decreased odds.")



Logistic regression results (odds ratios)

Number of obs = 36
Log likelihood = -9.974556
LR chi2(4) = 29.96
Prob > chi2 = 0.000005
Pseudo R2 = 0.6003

         Odds ratio  Std. err.      z  P>|z|  [95% CI lower]  [95% CI upper]
const      0.000001   7.379845 -1.959  0.050        0.000000        1.007163
HocVan     0.821097   0.272055 -0.725  0.469        0.481749        1.399484
Tuoi       1.480621   0.158766  2.472  0.013        1.084681        2.021092
ThuNhap    1.339061   0.154078  1.895  0.058        0.990032        1.811139
HoKhau     0.314050   1.005390 -1.152  0.249        0.043773        2.253136


Formatted Table:

| Variable   |   Odds ratio |   Std. err. |      z |   P>|z| |   [95% CI lower] |   [95% CI upper] |
|------------|--------------|-------------|--------|---------|------------------|------------------|
| const      |     1e-06    |    7.37985  | -1.959 |   0.05  |         0        |          1.00716 |
| HocVan     |     0.821097 |    0.272055 | -0.725 |   0.469 | 